# PySpark Getting Started - Eenvoudige Beginnersgids

Auteur: Claude Sonnet 4

## Wat is PySpark?

PySpark is de Python API voor Apache Spark, een krachtige engine voor data verwerking. Met PySpark kun je:

- **Grote datasets** verwerken op een efficiënte manier
- **SQL-achtige queries** uitvoeren op gestructureerde data  
- **Data transformaties** en aggregaties uitvoeren
- **Verschillende dataformaten** inlezen en exporteren
- **Snelle data analyse** doen met gedistribueerde computing

## Wat gaan we leren?

In deze **eenvoudige** tutorial leren we:
1. PySpark opzetten (lokaal, zonder complexe setup!)
2. CSV en JSON bestanden inladen
3. Data verkennen en transformeren
4. SQL queries uitvoeren met Spark SQL
5. Data aggregaties en grouping
6. Data opslaan in verschillende formaten

**✨ Geen database setup nodig - we werken met lokale bestanden!**

Let's get started! 🚀

## 1. Installatie en Setup

**Vereisten:**
- Python 3.8+
- Java 17+ (wordt automatisch gebruikt als beschikbaar)

**Installatie:**
```bash
pip install pyspark python-dotenv pandas
```

**Let's go! 🚀**

In [1]:
# Importeer alle benodigde libraries
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd

# Laad environment variabelen
load_dotenv()

print("✅ Alle packages succesvol geïmporteerd!")
print("� Klaar om te beginnen met PySpark!")
print("� SparkSession wordt aangemaakt in de volgende cel")

✅ Alle packages succesvol geïmporteerd!
� Klaar om te beginnen met PySpark!
� SparkSession wordt aangemaakt in de volgende cel


In [2]:
# Maak SparkSession met PostgreSQL ondersteuning
spark = SparkSession.builder \
    .appName("PySpark PostgreSQL Tutorial") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

# Zet log level naar WARN om minder verbose output te krijgen
spark.sparkContext.setLogLevel("WARN")

print("🎉 SparkSession succesvol gestart!")
print(f"💻 Spark UI beschikbaar op: {spark.sparkContext.uiWebUrl}")
print(f"📊 Spark versie: {spark.version}")
print("� Klaar om data te verwerken!")

:: loading settings :: url = jar:file:/Users/eva/Documents/Projects/Fruta/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/eva/.ivy2.5.2/cache
The jars for the packages stored in: /Users/eva/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6dfe50a0-42d6-4b7f-979e-f296b4e364b1;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 50ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	

🎉 SparkSession succesvol gestart!
💻 Spark UI beschikbaar op: http://prootje.fritz.box:4040
📊 Spark versie: 4.0.0
� Klaar om data te verwerken!


## 3. Verbinding maken met PostgreSQL

Nu gaan we verbinding maken met onze PostgreSQL database. We gebruiken de environment variabelen uit het `.env` bestand.

In [3]:
# Haal database credentials op uit environment variabelen
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD") 
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

# Bouw de JDBC URL
jdbc_url = f"jdbc:postgresql://{db_host}:{db_port}/{db_name}"

# Database verbinding eigenschappen
connection_properties = {
    "user": db_user,
    "password": db_password,
    "driver": "org.postgresql.Driver",
    "fetchsize": "1000"  # Optimalisatie voor betere performance
}

print("🔗 Database verbinding geconfigureerd")
print(f"📡 JDBC URL: {jdbc_url}")
print(f"👤 Gebruiker: {db_user}")
print(f"🏠 Host: {db_host}:{db_port}")
print(f"🗄️  Database: {db_name}")

🔗 Database verbinding geconfigureerd
📡 JDBC URL: jdbc:postgresql://localhost:5432/fruta
👤 Gebruiker: fruta
🏠 Host: localhost:5432
🗄️  Database: fruta


## 4. Data inladen uit PostgreSQL

Laten we eerst kijken welke tabellen er beschikbaar zijn in de database en vervolgens data inladen.

In [8]:
# Maak SparkSession met PostgreSQL ondersteuning en binding fix
try:
    # Stop eventuele bestaande SparkContext
    try:
        spark.stop()
        print("🧹 Oude SparkSession gestopt")
    except:
        pass
    
    # Maak nieuwe SparkSession met expliciete configuratie
    spark = SparkSession.builder \
        .appName("PySpark PostgreSQL Tutorial") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
        .config("spark.driver.bindAddress", "127.0.0.1") \
        .config("spark.driver.host", "127.0.0.1") \
        .config("spark.ui.port", "4040") \
        .getOrCreate()
    
    # Zet log level naar WARN om minder verbose output te krijgen
    spark.sparkContext.setLogLevel("WARN")
    
    print("🎉 SparkSession succesvol gestart!")
    print(f"💻 Spark UI beschikbaar op: {spark.sparkContext.uiWebUrl}")
    print(f"📊 Spark versie: {spark.version}")
    print("🚀 Klaar om data te verwerken!")
    
except Exception as e:
    print(f"❌ Error bij maken SparkSession: {e}")
    print("💡 Probeer de kernel te herstarten en opnieuw te beginnen")
    print("💡 Of wacht even en probeer opnieuw - mogelijk zijn poorten nog bezet")


🧹 Oude SparkSession gestopt
🎉 SparkSession succesvol gestart!
💻 Spark UI beschikbaar op: http://127.0.0.1:4040
📊 Spark versie: 4.0.0
🚀 Klaar om data te verwerken!


25/08/23 08:58:47 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 194407 ms exceeds timeout 120000 ms
25/08/23 08:58:47 WARN SparkContext: Killing executors is not supported by current scheduler.
25/08/23 08:58:51 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o

In [ ]:
# Functie om alle beschikbare tabellen te tonen
def show_available_tables():
    """
    Toont alle tabellen in de PostgreSQL database
    """
    try:
        # Query om alle tabellen in de public schema te krijgen
        tables_query = """
        SELECT table_name, table_type 
        FROM information_schema.tables 
        WHERE table_schema = 'public' 
        ORDER BY table_name
        """
        
        tables_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("query", tables_query) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("driver", "org.postgresql.Driver") \
            .load()
        
        print("📋 Beschikbare tabellen in de database:")
        print("=" * 50)
        tables_df.show(truncate=False)
        
        # Ook als Python list retourneren voor verder gebruik
        table_names = [row.table_name for row in tables_df.collect()]
        print(f"📊 Totaal aantal tabellen: {len(table_names)}")
        print(f"🏷️  Tabelnamen: {table_names}")
        
        return table_names
        
    except Exception as e:
        print(f"❌ Fout bij ophalen tabellen: {e}")
        return []

# Roep de functie aan om alle tabellen te zien
available_tables = show_available_tables()

📋 Beschikbare tabellen in de database:
+----------+----------+
|table_name|table_type|
+----------+----------+
|notes     |BASE TABLE|
|profiles  |BASE TABLE|
|records   |BASE TABLE|
|scenarios |BASE TABLE|
+----------+----------+

📊 Totaal aantal tabellen: 4
🏷️  Tabelnamen: ['notes', 'profiles', 'records', 'scenarios']


In [5]:
# Functie om een volledige tabel in te laden
def load_table(table_name, limit=None):
    """
    Laadt een tabel uit PostgreSQL in een Spark DataFrame
    
    Args:
        table_name (str): Naam van de tabel
        limit (int, optional): Beperkt het aantal rijen
    
    Returns:
        DataFrame: Spark DataFrame met de tabeldata
    """
    try:
        query = f"SELECT * FROM {table_name}"
        if limit:
            query += f" LIMIT {limit}"
            
        df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("query", query) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("driver", "org.postgresql.Driver") \
            .load()
        
        print(f"✅ Tabel '{table_name}' succesvol ingeladen!")
        print(f"📊 Aantal rijen: {df.count()}")
        print(f"📋 Aantal kolommen: {len(df.columns)}")
        
        return df
    except Exception as e:
        print(f"❌ Fout bij inladen tabel '{table_name}': {e}")
        return None

# Probeer een tabel in te laden (vervang 'your_table_name' met een echte tabel)
df = load_table("records", limit=1000)
# print("💡 Gebruik load_table('table_name') om een specifieke tabel in te laden")

✅ Tabel 'records' succesvol ingeladen!
📊 Aantal rijen: 1000
📋 Aantal kolommen: 7


## 5. DataFrame Basics - Data verkennen

Nu we weten hoe we data kunnen inladen, gaan we leren hoe we met Spark DataFrames werken. Een **DataFrame** is een gedistribueerde verzameling data georganiseerd in benoemde kolommen.

In [ ]:
# Laten we een voorbeeld DataFrame maken om de basics te demonstreren
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from datetime import date

# Definieer het schema
schema = StructType([
    StructField("naam", StringType(), True),
    StructField("leeftijd", IntegerType(), True),
    StructField("stad", StringType(), True),
    StructField("salaris", DoubleType(), True),
    StructField("startdatum", DateType(), True)
])

# Maak voorbeelddata
sample_data = [
    ("Anna", 28, "Amsterdam", 65000.0, date(2022, 1, 15)),
    ("Bob", 34, "Rotterdam", 72000.0, date(2021, 3, 22)),
    ("Charlie", 29, "Utrecht", 58000.0, date(2023, 5, 10)),
    ("Diana", 31, "Den Haag", 69000.0, date(2022, 8, 5)),
    ("Eva", 26, "Eindhoven", 61000.0, date(2023, 2, 18)),
    ("Frank", 35, "Groningen", 75000.0, date(2020, 11, 30)),
    ("Grace", 30, "Amsterdam", 67000.0, date(2022, 4, 12)),
    ("Henry", 33, "Rotterdam", 71000.0, date(2021, 9, 8))
]

# Maak DataFrame
df_employees = spark.createDataFrame(sample_data, schema)

print("🎯 Voorbeeld DataFrame 'employees' aangemaakt!")
print("📊 Dit is hoe een Spark DataFrame eruit ziet:")
df_employees.show()

In [ ]:
# Basis DataFrame operaties voor data exploratie

print("🔍 DATAFRAME EXPLORATIE")
print("=" * 50)

# 1. Schema bekijken
print("📋 Schema (datatypes van kolommen):")
df_employees.printSchema()

print("\n" + "=" * 50)

# 2. Aantal rijen en kolommen
print(f"📊 Aantal rijen: {df_employees.count()}")
print(f"📋 Aantal kolommen: {len(df_employees.columns)}")
print(f"🏷️  Kolomnamen: {df_employees.columns}")

print("\n" + "=" * 50)

# 3. Eerste paar rijen bekijken
print("👁️  Eerste 3 rijen:")
df_employees.show(3)

print("=" * 50)

# 4. Beschrijvende statistieken
print("📈 Beschrijvende statistieken:")
df_employees.describe().show()

print("=" * 50)

# 5. Unieke waarden in een kolom
print("🏙️  Unieke steden:")
df_employees.select("stad").distinct().show()

## 6. DataFrame Transformaties

**Transformaties** zijn lazy operations die een nieuw DataFrame retourneren. Ze worden pas uitgevoerd wanneer een **actie** wordt aangeroepen.

### Belangrijke transformaties:
- `select()` - selecteer specifieke kolommen
- `filter()` / `where()` - filter rijen op basis van voorwaarden  
- `withColumn()` - voeg nieuwe kolom toe of wijzig bestaande
- `groupBy()` - groepeer data voor aggregaties
- `orderBy()` / `sort()` - sorteer data
- `join()` - voeg DataFrames samen

In [ ]:
# 1. SELECT - specifieke kolommen selecteren
print("🎯 SELECT transformatie:")
df_names_salaries = df_employees.select("naam", "salaris", "stad")
df_names_salaries.show()

print("\n" + "=" * 50)

# 2. FILTER/WHERE - rijen filteren op voorwaarden  
print("🔍 FILTER transformatie - salaris > 65000:")
df_high_earners = df_employees.filter(df_employees.salaris > 65000)
df_high_earners.show()

print("\n" + "=" * 50)

# 3. Meerdere voorwaarden combineren
print("🔍 FILTER met meerdere voorwaarden - Amsterdam EN salaris > 60000:")
df_amsterdam_high = df_employees.filter(
    (df_employees.stad == "Amsterdam") & 
    (df_employees.salaris > 60000)
)
df_amsterdam_high.show()

print("\n" + "=" * 50)

# 4. String operations
print("🔤 String filtering - namen die beginnen met 'A':")
df_a_names = df_employees.filter(df_employees.naam.startswith("A"))
df_a_names.show()

In [ ]:
# 5. WITH_COLUMN - nieuwe kolommen toevoegen of wijzigen
print("➕ WITH_COLUMN transformatie - nieuwe kolommen toevoegen:")

# Voeg nieuwe kolommen toe
df_enhanced = df_employees \
    .withColumn("salaris_per_maand", col("salaris") / 12) \
    .withColumn("is_senior", col("leeftijd") >= 30) \
    .withColumn("naam_upper", upper(col("naam"))) \
    .withColumn("jaren_ervaring", 2025 - year(col("startdatum")))

df_enhanced.select("naam", "salaris", "salaris_per_maand", "is_senior", "jaren_ervaring").show()

print("\n" + "=" * 50)

# 6. ORDER_BY - data sorteren
print("📊 ORDER_BY transformatie - sorteren op salaris (aflopend):")
df_sorted = df_employees.orderBy(col("salaris").desc())
df_sorted.show()

print("\n" + "=" * 50)

# 7. Sorteren op meerdere kolommen
print("📊 Sorteren op stad (oplopend) en salaris (aflopend):")
df_multi_sort = df_employees.orderBy(col("stad").asc(), col("salaris").desc())
df_multi_sort.show()

## 7. Aggregaties en Groepering

Aggregaties zijn krachtige operaties om statistieken te berekenen over groepen data.

In [ ]:
# Eenvoudige aggregaties op hele DataFrame
print("📈 EENVOUDIGE AGGREGATIES:")
print("=" * 50)

# Basis statistieken
df_employees.agg(
    avg("salaris").alias("gemiddeld_salaris"),
    max("salaris").alias("max_salaris"), 
    min("salaris").alias("min_salaris"),
    count("*").alias("aantal_werknemers")
).show()

print("\n" + "=" * 50)

# GROUP BY aggregaties
print("👥 GROUP BY aggregaties - per stad:")
df_city_stats = df_employees.groupBy("stad").agg(
    count("*").alias("aantal_werknemers"),
    avg("salaris").alias("gemiddeld_salaris"),
    max("salaris").alias("max_salaris"),
    min("leeftijd").alias("jongste_leeftijd"),
    max("leeftijd").alias("oudste_leeftijd")
).orderBy(col("gemiddeld_salaris").desc())

df_city_stats.show()

print("\n" + "=" * 50)

# Meer complexe aggregaties
print("💡 Geavanceerde aggregaties:")
df_complex = df_employees \
    .withColumn("salaris_categorie", 
                when(col("salaris") >= 70000, "Hoog")
                .when(col("salaris") >= 60000, "Gemiddeld")
                .otherwise("Laag")) \
    .groupBy("stad", "salaris_categorie") \
    .count() \
    .orderBy("stad", "salaris_categorie")

df_complex.show()

## 8. Spark SQL - SQL queries gebruiken

Een van de krachtigste features van Spark is de mogelijkheid om SQL queries te gebruiken op DataFrames!

In [ ]:
# DataFrame registreren als tijdelijke view/tabel
df_employees.createOrReplaceTempView("employees")

print("🗄️  DataFrame geregistreerd als 'employees' tabel")
print("=" * 50)

# Eenvoudige SQL query
print("📝 Eenvoudige SELECT query:")
result1 = spark.sql("""
    SELECT naam, stad, salaris 
    FROM employees 
    WHERE salaris > 65000
    ORDER BY salaris DESC
""")
result1.show()

print("\n" + "=" * 50)

# Complexere SQL query met aggregaties
print("📊 SQL query met aggregaties:")
result2 = spark.sql("""
    SELECT 
        stad,
        COUNT(*) as aantal_werknemers,
        ROUND(AVG(salaris), 2) as gemiddeld_salaris,
        MAX(salaris) as max_salaris,
        MIN(leeftijd) as jongste_leeftijd
    FROM employees 
    GROUP BY stad
    HAVING COUNT(*) >= 1
    ORDER BY gemiddeld_salaris DESC
""")
result2.show()

print("\n" + "=" * 50)

# Window functions in SQL
print("🪟 SQL met Window Functions:")
result3 = spark.sql("""
    SELECT 
        naam,
        stad,
        salaris,
        ROW_NUMBER() OVER (PARTITION BY stad ORDER BY salaris DESC) as rang_in_stad,
        AVG(salaris) OVER (PARTITION BY stad) as gemiddeld_salaris_stad
    FROM employees
    ORDER BY stad, rang_in_stad
""")
result3.show()

## 9. Werken met echte PostgreSQL data

Nu gaan we terug naar onze PostgreSQL database en echte data analyseren!

In [ ]:
# Bekijk opnieuw welke tabellen beschikbaar zijn
print("🔍 Beschikbare tabellen in de database:")
available_tables = show_available_tables()

print("\n" + "=" * 60)

# Aangepaste functie voor directe SQL queries op database
def execute_db_query(query, limit=None):
    """
    Voert een directe SQL query uit op de PostgreSQL database
    
    Args:
        query (str): SQL query om uit te voeren
        limit (int, optional): Beperkt het aantal rijen
    
    Returns:
        DataFrame: Spark DataFrame met de query resultaten
    """
    try:
        if limit and "LIMIT" not in query.upper():
            query += f" LIMIT {limit}"
            
        df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("query", query) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("driver", "org.postgresql.Driver") \
            .option("fetchsize", "1000") \
            .load()
        
        return df
    except Exception as e:
        print(f"❌ Fout bij uitvoeren query: {e}")
        return None

# Voorbeeld: tel rijen in elke tabel (vervang met echte tabelnamen)
print("💡 Gebruik execute_db_query() om directe SQL queries uit te voeren")
print("💡 Gebruik load_table() om volledige tabellen in te laden")

# Voorbeeld van hoe je zou kunnen werken met echte data:
# df_real_data = execute_db_query("SELECT * FROM your_table_name", limit=100)
# if df_real_data:
#     df_real_data.show()
#     df_real_data.printSchema()

## 10. Performance Tips en Best Practices

### 🚀 Performance Tips:

1. **Partitionering**: Gebruik `partitionColumn` bij het inladen van grote tabellen
2. **Fetchsize**: Stel `fetchsize` in voor optimale JDBC performance
3. **Predicaat pushdown**: Filters worden automatisch naar de database gepusht
4. **Caching**: Cache DataFrames die je meerdere keren gebruikt
5. **Lazy evaluation**: Transformaties worden pas uitgevoerd bij een actie

### 💡 Best Practices:

1. **Schema inference**: Definieer schema's expliciet voor betere performance
2. **Column pruning**: Selecteer alleen de kolommen die je nodig hebt
3. **Resource management**: Stop SparkSessions om resources vrij te geven

In [ ]:
# Performance optimalisaties

print("⚡ PERFORMANCE OPTIMALISATIES")
print("=" * 50)

# 1. Caching van DataFrames die je meerdere keren gebruikt
print("💾 Caching van DataFrames:")
df_employees.cache()  # Cache het DataFrame in geheugen
print("✅ DataFrame gecacht voor hergebruik")

# 2. Explain query execution plan
print("\n📋 Query execution plan:")
df_employees.filter(col("salaris") > 65000).explain()

print("\n" + "=" * 50)

# 3. Optimalisatie voor grote PostgreSQL tabellen
def load_table_optimized(table_name, partition_column=None, limit=None):
    """
    Geoptimaliseerde functie voor het laden van grote tabellen
    """
    options = {
        "url": jdbc_url,
        "dbtable": table_name,
        "user": db_user,
        "password": db_password,
        "driver": "org.postgresql.Driver",
        "fetchsize": "10000"  # Grotere fetchsize voor performance
    }
    
    # Voeg partitionering toe voor zeer grote tabellen
    if partition_column:
        options.update({
            "partitionColumn": partition_column,
            "lowerBound": "1",
            "upperBound": "1000000",
            "numPartitions": "4"
        })
    
    query = f"(SELECT * FROM {table_name}"
    if limit:
        query += f" LIMIT {limit}"
    query += ") as subquery"
    
    options["dbtable"] = query
    
    return spark.read.format("jdbc").options(**options).load()

print("🎯 Geoptimaliseerde load functie gedefinieerd")

print("\n" + "=" * 50)

# 4. Memory management
print("💾 Memory usage info:")
print(f"📊 Spark UI: {spark.sparkContext.uiWebUrl}")
print("💡 Bekijk de Spark UI voor gedetailleerde performance metrics")

## 11. Data Export en Opslaan

PySpark kan data in verschillende formaten opslaan:

In [ ]:
# Data opslaan in verschillende formaten

print("💾 DATA EXPORT MOGELIJKHEDEN")
print("=" * 50)

# Maak een output directory
import os
output_dir = "/Users/eva/Documents/Projects/Fruta/output/spark_export"
os.makedirs(output_dir, exist_ok=True)

# 1. CSV export
print("📄 Opslaan als CSV:")
df_summary = df_employees.groupBy("stad").agg(
    count("*").alias("aantal"), 
    avg("salaris").alias("gem_salaris")
)

csv_path = f"{output_dir}/employees_summary.csv"
df_summary.coalesce(1).write.mode("overwrite").option("header", "true").csv(csv_path)
print(f"✅ CSV opgeslagen in: {csv_path}")

print("\n" + "=" * 50)

# 2. JSON export  
print("📋 Opslaan als JSON:")
json_path = f"{output_dir}/employees_summary.json"
df_summary.coalesce(1).write.mode("overwrite").json(json_path)
print(f"✅ JSON opgeslagen in: {json_path}")

print("\n" + "=" * 50)

# 3. Parquet export (geoptimaliseerd formaat)
print("🗜️  Opslaan als Parquet:")
parquet_path = f"{output_dir}/employees_summary.parquet"
df_summary.write.mode("overwrite").parquet(parquet_path)
print(f"✅ Parquet opgeslagen in: {parquet_path}")

print("\n" + "=" * 50)

# 4. Terug naar PostgreSQL schrijven (voorbeeld)
print("🗄️  Terug naar database schrijven:")
print("💡 Voorbeeld code voor het schrijven naar een nieuwe tabel:")
print("""
df_summary.write \\
    .format("jdbc") \\
    .option("url", jdbc_url) \\
    .option("dbtable", "employees_summary") \\
    .option("user", db_user) \\
    .option("password", db_password) \\
    .option("driver", "org.postgresql.Driver") \\
    .mode("overwrite") \\
    .save()
""")

print("\n" + "=" * 50)

# 5. Naar Pandas DataFrame converteren (voor kleine datasets)
print("🐼 Conversie naar Pandas DataFrame:")
pandas_df = df_summary.toPandas()
print("✅ Geconverteerd naar Pandas DataFrame")
print(f"📊 Shape: {pandas_df.shape}")
print(pandas_df.head())

## 12. Samenvatting en Volgende Stappen

### 🎉 Gefeliciteerd! Je hebt de PySpark basics geleerd:

✅ **SparkSession** opzetten en configureren  
✅ **PostgreSQL verbinding** maken met JDBC  
✅ **DataFrames** maken en manipuleren  
✅ **Transformaties** toepassen (select, filter, groupBy, etc.)  
✅ **Aggregaties** uitvoeren voor data analyse  
✅ **Spark SQL** gebruiken voor complexe queries  
✅ **Performance optimalisaties** toepassen  
✅ **Data export** naar verschillende formaten  

### 🚀 Volgende stappen om verder te leren:

1. **Machine Learning**: Verken MLlib voor gedistribueerde ML
2. **Streaming**: Leer Structured Streaming voor real-time data
3. **Advanced SQL**: Window functions, joins, CTEs
4. **Cluster deployment**: Deploy op echte Spark clusters
5. **Delta Lake**: Versioned data lakes met ACID properties

### 📚 Handige resources:

- [PySpark Documentatie](https://spark.apache.org/docs/latest/api/python/)
- [Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [MLlib Guide](https://spark.apache.org/docs/latest/ml-guide.html)

In [ ]:
# Cleanup - Stop de SparkSession om resources vrij te geven
print("🧹 CLEANUP")
print("=" * 50)

# Uncache alle DataFrames
df_employees.unpersist()
print("✅ DataFrames uncached")

# Optioneel: Stop de SparkSession
# Uncomment de volgende regel als je klaar bent
# spark.stop()
# print("✅ SparkSession gestopt")

print("🎯 Tutorial voltooid!")
print("💡 Je kunt nu experimenteren met je eigen PostgreSQL data")
print("🚀 Veel succes met PySpark!")